# 02 — Exploratory Environmental Analysis

**Project:** Brisbane River Water Quality Analysis

## Purpose

This notebook explores the *cleaned* dataset produced by Notebook 1
(`data/processed/water_quality_cleaned.csv`) to answer six research questions about the
Brisbane River's water quality over the ~11-month monitoring period (Aug 2023 – Jun 2024).

**Research questions**

- **RQ1** — What are the typical water-quality conditions?
- **RQ2** — Which water-quality parameters show the greatest variability?
- **RQ3** — How does water quality change over time?
- **RQ4** — Are there identifiable monthly or seasonal patterns?
- **RQ5** — What relationships exist between water-quality parameters?
- **RQ6** — Are there periods of unusual water-quality conditions?

## A note on interpretation

This is an **exploratory, observational** analysis of sensor data — it can show that
parameters are *associated with* time of year, time of day, or each other, but it cannot
establish *why* on its own. Wording throughout uses "associated with", "correlated with",
"may indicate", and "warrants further investigation" rather than causal language, and every
finding is paired with its environmental relevance and its limits.

**Prerequisite:** run `notebooks/01_data_understanding_cleaning.ipynb` first so that
`data/processed/water_quality_cleaned.csv` exists.


## 1. Imports and setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

CLEANED_PATH = Path('../data/processed/water_quality_cleaned.csv')

# The key continuous water-quality parameters analysed throughout this notebook
KEY_PARAMS = ['temperature', 'dissolved_oxygen', 'dissolved_oxygen_pct_sat', 'ph',
              'salinity', 'specific_conductance', 'turbidity', 'chlorophyll', 'water_speed']

SEASON_ORDER = ['Spring', 'Summer', 'Autumn', 'Winter']


## 2. Load cleaned dataset

In [ ]:
df = pd.read_csv(CLEANED_PATH, parse_dates=['timestamp'])
df['season'] = pd.Categorical(df['season'], categories=SEASON_ORDER, ordered=True)
print(f'Loaded {len(df):,} rows, {df["timestamp"].min()} to {df["timestamp"].max()}')
df.head()


## RQ1 — What are the typical water-quality conditions?

We start with basic descriptive statistics for each key parameter: mean, median, standard
deviation, and range. The median is a useful companion to the mean here because several
parameters (e.g. turbidity) are right-skewed by occasional high-flow events.

In [ ]:
desc = df[KEY_PARAMS].describe().T
desc['median'] = df[KEY_PARAMS].median()
desc = desc[['count', 'mean', 'median', 'std', 'min', 'max']].round(3)
desc


**Environmental relevance:** these figures describe the "baseline" condition of the
river over the monitoring period — e.g. the typical temperature, pH, and dissolved oxygen
level a healthy reading should be compared against. Large gaps between mean and median, or a
wide min–max range, are early signals of skew that RQ2 examines more formally.

**Limitation:** these are simple summary statistics over the whole period; they say nothing
yet about *when* extreme values occur (covered in RQ3, RQ4, RQ6) or how reliable each mean is
given the uneven missingness documented in Notebook 1 (`count` above shows how many non-null
readings each statistic is based on).

## RQ2 — Which water-quality parameters show the greatest variability?

Raw standard deviation isn't comparable across parameters measured in different units and
scales (e.g. pH ranges ~1 unit, turbidity ranges tens of NTU). We use the **coefficient of
variation** (CV = std / mean) to compare relative variability on a common, unit-free scale.

In [ ]:
cv = (df[KEY_PARAMS].std() / df[KEY_PARAMS].mean()).sort_values(ascending=False)
cv_table = cv.to_frame('coefficient_of_variation').round(3)
cv_table


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cv.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_xlabel('Coefficient of variation (std / mean)')
ax.set_title('Relative variability by parameter')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for ax, col in zip(axes, KEY_PARAMS):
    sns.boxplot(y=df[col].dropna(), ax=ax, color='#55A868')
    ax.set_title(col)
    ax.set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.flatten()
for ax, col in zip(axes, KEY_PARAMS):
    sns.histplot(df[col].dropna(), bins=40, ax=ax, color='#C44E52', kde=True)
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


**Environmental relevance:** parameters with high CV (typically water speed and
turbidity on a river) are the ones most sensitive to short-term hydrological events — tidal
flow changes, rainfall/runoff, or boat wake — while parameters with low CV (typically pH and
temperature) are more tightly buffered by the river's chemistry and thermal mass. This is
useful for prioritising which parameters warrant closer monitoring for sudden change.

**Limitation:** CV is a whole-period summary. A parameter could have moderate overall CV but
still contain a short, sharp anomalous spike — that's investigated directly in RQ6.

## RQ3 — How does water quality change over time?

We resample to a daily mean per parameter and overlay a 7-day rolling average to separate
short-term noise from the underlying trend across the ~11-month record.

In [ ]:
daily = df.set_index('timestamp')[KEY_PARAMS].resample('D').mean()

fig, axes = plt.subplots(len(KEY_PARAMS), 1, figsize=(13, 3 * len(KEY_PARAMS)), sharex=True)
for ax, col in zip(axes, KEY_PARAMS):
    ax.plot(daily.index, daily[col], color='lightgray', linewidth=0.8, label='daily mean')
    ax.plot(daily.index, daily[col].rolling(7, min_periods=1).mean(), color='#4C72B0', linewidth=1.6, label='7-day rolling mean')
    ax.set_ylabel(col)
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Date')
fig.suptitle('Daily mean and 7-day rolling trend by parameter', y=1.001)
plt.tight_layout()
plt.show()


**Environmental relevance:** the 7-day rolling trend line makes seasonal-scale movement
visible (e.g. temperature's expected annual cycle) while damping single-day noise from tides
or short events. Any sustained rise or fall in the rolling mean over weeks-to-months is a
change *worth investigating* — it may be seasonal (see RQ4), a slow sensor drift (see
Notebook 1's missingness findings, since drift often precedes an outage), or an environmental
shift.

**Limitation:** the daily-mean line will show visible gaps or dips wherever Notebook 1's
missing-value analysis found reduced sensor coverage — a thin/gappy stretch of the gray line
reflects data availability, not necessarily a real change in the river.

## RQ4 — Are there identifiable monthly or seasonal patterns?

Brisbane is in the Southern Hemisphere, so **Summer = Dec–Feb** and **Winter = Jun–Aug**
(engineered in Notebook 1 as the `season` column).

In [ ]:
monthly_stats = df.groupby('month')[KEY_PARAMS].mean().round(2)
monthly_stats


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(17, 12))
axes = axes.flatten()
for ax, col in zip(axes, KEY_PARAMS):
    sns.boxplot(data=df, x='season', y=col, order=SEASON_ORDER, ax=ax, palette='Set2')
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


**Environmental relevance:** water temperature is expected to track the seasonal air
temperature cycle closely (a useful sanity check that the sensor and season labels are
correct); dissolved oxygen typically moves *inversely* to temperature because colder water
holds more dissolved gas — this expected inverse relationship is examined directly under RQ5.
Seasonal salinity/conductance shifts can reflect wet-season freshwater inflow diluting the
estuarine section of the river versus drier months.

**Limitation:** the monitoring period covers **less than one full year (~11 months)**, so
this is one partial seasonal cycle, not multiple years of seasonal data — patterns seen here
are a first look, not a confirmed multi-year seasonal signal.

### Daily (hour-of-day) pattern

Some parameters — particularly dissolved oxygen — are expected to show a **diel (24-hour)
cycle** driven by photosynthesis: oxygen produced by aquatic plants/algae during daylight,
consumed by respiration overnight.

In [ ]:
hourly = df.groupby('hour')[['dissolved_oxygen', 'dissolved_oxygen_pct_sat', 'chlorophyll', 'ph']].mean()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flatten(), hourly.columns):
    ax.plot(hourly.index, hourly[col], marker='o', color='#8172B2')
    ax.set_title(f'Mean {col} by hour of day')
    ax.set_xlabel('Hour (0-23)')
plt.tight_layout()
plt.show()


**Environmental relevance:** if dissolved oxygen and pH peak during daylight hours and
dip overnight, this is consistent with the expected photosynthesis/respiration diel cycle —
a sign of active biological productivity in the river rather than a sensor artifact.

**Limitation:** hour-of-day is averaged across all seasons and daylight lengths vary across
the monitoring period, so the exact peak hour is an approximation, not a precise sunrise/
sunset-aligned measurement.

## RQ5 — What relationships exist between water-quality parameters?

We use Pearson correlation across the key parameters, then look closely at the
temperature–dissolved oxygen relationship, which has a well-established physical basis
(colder water holds more dissolved gas).

In [ ]:
corr = df[KEY_PARAMS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation matrix — key water-quality parameters')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sample = df.sample(min(5000, len(df)), random_state=42)
sns.scatterplot(data=sample, x='temperature', y='dissolved_oxygen', hue='season',
                 hue_order=SEASON_ORDER, alpha=0.4, s=15, ax=ax, palette='Set2')
ax.set_title('Temperature vs Dissolved Oxygen')
plt.tight_layout()
plt.show()

r = df[['temperature', 'dissolved_oxygen']].corr().iloc[0, 1]
print(f'Pearson correlation (temperature, dissolved_oxygen): {r:.3f}')


**Environmental relevance:** a negative correlation between temperature and dissolved
oxygen is **associated with** the known physical relationship between water temperature and
gas solubility — warmer water can hold less dissolved oxygen. Where the correlation matrix
shows other strong pairs (e.g. salinity and specific conductance, which are physically linked
since conductance is partly driven by dissolved ions), that reinforces internal consistency
of the sensor readings.

**Limitation:** correlation does not establish causation, and several parameters here share
common external drivers (e.g. season, tide, rainfall) that could produce correlated movement
without one directly causing the other. Any relationship identified here **warrants further
investigation** with more targeted data (e.g. rainfall records) before drawing firm
conclusions.

## RQ6 — Are there periods of unusual water-quality conditions?

We flag readings that fall more than 3 standard deviations from that parameter's overall mean
as *statistically* unusual, then look at *when* they cluster in time. As established in
Notebook 1, such values are **not automatically removed** — they may represent genuine
environmental events (storms, high-flow periods) rather than errors.

In [ ]:
def flag_extreme(series, n_std=3):
    mean, std = series.mean(), series.std()
    return (series - mean).abs() > n_std * std

extreme_flags = pd.DataFrame({col: flag_extreme(df[col]) for col in KEY_PARAMS})
extreme_counts = extreme_flags.sum().sort_values(ascending=False)
extreme_counts.to_frame('extreme_reading_count')


In [ ]:
# Which parameter has the most extreme-flagged readings? Show its extreme periods over time.
top_param = extreme_counts.idxmax()
extreme_dates = df.loc[extreme_flags[top_param], 'timestamp']

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df['timestamp'], df[top_param], color='lightgray', linewidth=0.6, label=top_param)
ax.scatter(extreme_dates, df.loc[extreme_flags[top_param], top_param],
           color='#C44E52', s=12, label='flagged as extreme (>3 std)', zorder=5)
ax.set_title(f'{top_param}: flagged extreme readings over time')
ax.legend()
plt.tight_layout()
plt.show()

print(f'{top_param}: {extreme_flags[top_param].sum()} extreme readings')
print('Date range of extreme readings:', extreme_dates.min(), 'to', extreme_dates.max())


**Environmental relevance:** if flagged extreme readings **cluster** in a short window
rather than scattering randomly across the whole period, that clustering **may indicate** a
real, time-bounded event — a storm/high-flow period for turbidity or water speed, for
example — which is exactly the kind of event an environmental monitoring program is designed
to catch, not an artifact to discard.

**Limitation:** a simple 3-standard-deviation threshold is a statistical convenience, not an
environmental threshold — it does not by itself confirm an ecological or water-quality
incident. Confirming that would require cross-referencing external records (e.g. Bureau of
Meteorology rainfall data for Brisbane) which is outside the scope of this dataset, and is
listed as a recommendation for further investigation in Notebook 3.

## What to take away from this notebook

- **RQ1–RQ2**: the dataset has a well-defined typical range for each parameter, with water
  speed and turbidity the most relatively variable — consistent with their sensitivity to
  short-term flow/weather events, and pH/temperature the most stable.
- **RQ3–RQ4**: parameters show visible movement across the ~11-month record and across
  season/month/hour groupings, but this is a single partial seasonal cycle — not enough to
  confirm a repeating annual pattern.
- **RQ5**: temperature and dissolved oxygen are associated in the direction physically
  expected (inverse relationship); other correlated pairs are consistent with known chemical/
  physical relationships between the parameters, not proof of a causal link.
- **RQ6**: extreme readings exist for every parameter but are not removed; where they cluster
  in time, they are flagged as periods worth further investigation rather than treated as
  noise.
- Every finding above is phrased as an association, not a proven cause — Notebook 3 carries
  forward only the strongest, clearest of these findings into a small set of presentation
  visualisations with the same care around limitations.
